In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
print("Hello ML")

In [ ]:
housing_full = pd.read_csv(Path("datasets/housing/housing.csv"))

In [ ]:
housing_full.head()

In [ ]:
housing_full.info()

In [ ]:
housing_full.describe()

In [ ]:
housing_full.hist(bins=50, figsize=(12, 8))
plt.show()

In [ ]:
housing_full["income_cat"] = pd.cut(housing_full["median_income"],
                                    bins=[0,1.5,3.0,4.5,6, np.inf],
                                    labels=[1,2,3,4,5])

cat_counts = housing_full['income_cat'].value_counts().sort_index()
cat_counts.plot.bar(rot=0, grid = True)
plt.xlabel("Income category")
plt.ylabel("No of districts")
plt.show()
print(cat_counts[2]*0.2)

In [ ]:
#Random sampling into train and test set:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split( housing_full, test_size=0.2, random_state=42)

In [ ]:
test_set["income_cat"].value_counts()

In [ ]:
#Stratified sampling into train and test set:
strat_train_set, strat_test_set = train_test_split( housing_full, test_size=0.2, random_state=42, stratify=housing_full["income_cat"])

In [ ]:
strat_test_set["income_cat"].value_counts()

Now after spliting we can delete income_cat column from splits cause it can make our model learn extra noise and overfit th data.

As we original data doesn't contain categories and we aren't using input data to find category and give it to model.

so drop the column

In [ ]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

As in cell 16 we catogorized whole dataset and also printed no of districts of category 2 and in two different sampling and spliting methods random and stratified(median income) we got train set and test set.

So the value printed in cell 16 is 20% of no of districts belonging to 2nd category = 1316.2

And we extracted exact values of eac categories in test set derived from both ways(random and stratified(median income)).

We can see in random (2nd cat) = 1269
while in stratified (end cat) = 1316

So closer value to 1316.2 is in stratified sampling so its important to have train and test sets stratified means having similar data distribution/

Conclusion: we always you well distributed and stratified data to get good test instances which can train and validate our model in all aspects.

In [ ]:
#Visualize geographical data. alpha helps to focus on denser regions.
housing = strat_train_set.copy()

housing.plot(kind="scatter", x="longitude", y="latitude", grid=True, alpha=0.2)
plt.show()

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True,
            s=housing["population"] / 100, label="population",
            c="median_house_value", cmap="inferno", colorbar=True,
            legend=True, sharex=False, figsize=(10,7))
plt.show()

We can see prime locations are BAY, costal and lowers regions, we can say that housing prices are dependent on location and population density. We need to find these patterns and Correlations.

In [ ]:
corr_matrix = housing.corr(numeric_only=True)

corr_matrix["median_house_value"].sort_values(ascending=False)

Found correlation coefficient (pearson's r) of each attribute with median house value, to get relation and it varies from -1 to 1. Closer to 1 means housing value increase as the respective atrribute increases. Closer to -1 means housing valuse decreases as the respective value increases.

In [ ]:
from pandas.plotting import scatter_matrix

attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]

scatter_matrix(housing[attributes], figsize=(12,8))
plt.show()

From Correlation coefficient and scattered plots of each attributes we found median income with highest positive coefficient and also it's graph shows linear relation as housing_values increase with increasing income.

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.5, grid=True)
plt.show()

*We* will also look for some new attributes which can be created like rooms_per_house, bedroom_ratio ,bedrooms_per_house, people_per_house and compare their correlation factors.

In [ ]:
housing["rooms_per_house"] = housing["total_rooms"] / housing["households"]
housing["bedroom_per_house"] = housing["total_bedrooms"] / housing["households"]
housing["bedroom_ratio"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["people_per_house"] = housing["population"] / housing["households"]

new_corr_matrix = housing.corr(numeric_only=True)
new_corr_matrix["median_house_value"].sort_values(ascending=False)

Rooms per house and bedroom ratio has good correlation factor than total rooms and bedrooms. These insights are helpfull.

In [ ]:
#Now cleaning process starts to prepare data for ml aogorithms.

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

#Seperated the labels and instances.

Data cleaning data usually means making data reliable and consumable by ml algorithms.
1. Handling missing values.
2. Removing Duplicates.
3. Fix Incorrect data.
4. Handle outliers.
5. Standardize formats like fonts, styles, upper/lowercase, making similar to avoid confusion.
6. Feature Engineering accumlating data two or more attributtes to make more reliable and effective attribute/s. Reduces data complexity and unwanted features.
7. Convert Categorical Data. ml needs numaerical data, so attributes are encoded into numbers or can be categotised.

In [ ]:
# We need to handle the total_bedrooms missing value so we using imputation method(fil null values with median of the attribute).

median = housing_full["total_bedrooms"].median()
housing["total_bedrooms"] = housing["total_bedrooms"].fillna(median)


In [ ]:
# We can chek if any other null values present.
housing.info()
# Perfect no null values present.

The above is usefull but we need to apply it on all datasets to handle missing values. So instead we can ue SimpleImputer class from scikit learn and create a instance and use it to handle any dataset, any attribute.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

In [ ]:
#But we can only use it on numerical values so extract only num values from train_set

housing_num = housing.select_dtypes(include=np.number)

#No fit the imouter to the new traing data, it will fill statistic values of each attribute
imputer.fit(housing_num)

imputer.statistics_


In [ ]:
# Now replace null with median of all attributes of each instance.

X = imputer.transform(housing_num)

#Sklearn outputs are always np.arrays so convert to pd.dataframes

housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)

Now we will handle text values/attributes cause ml mostly works on numbers so convert them to numbers. We similar attribute which is "ocean_proximity" it can be categorized as seen all instances have values among these: ['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN']

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(10)

We can use OrdinalEncoder to number the categories and replace eaach category with respective number in dataset but these numbers can be manipulative to ml as it can learn unneccessary patterns/noise like 1+2 = 3 so adding two categories we get third. So to avoid overfitting we will use OneHotEncoder which will use binary numbers(0,1) the category of instance will be marked as 1 and rest as 0, samee for all instances.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()

housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

#we get array of bits for each instance of size 5=no of total categories.

housing_cat_1hot.toarray()

In [ ]:
#List of categories
cat_encoder.categories_

In [ ]:
#We got a sparsed matrix but to maintain good readability we must have specifing columns and notations so a dataframe si best way:

cat_encoder.feature_names_in_ # Shows which attribute our encoder is working on

In [ ]:
columns = cat_encoder.get_feature_names_out() # Shows all category names
print(columns)

In [ ]:
housing_cat_1hot_output = pd.DataFrame(housing_cat_1hot.toarray(), columns=columns, index=housing.index)
housing_cat_1hot_output.head(5)

In [ ]:
print(type(housing_num)) # Only numeric_values
print(type(housing)) # Training set without median_house_value_column and also some missing values
print(type(housing_tr)) # Training set without any missing values and median_house_value_column
print(type(housing_cat_1hot_output)) # Categories dataframe
print(type(housing_cat_1hot))# Categories sparse matrix

Feature Scaling : We should scale our feature cause they vary a lot in units examole median income ranges 0-15 (tens unit) and total_rooms ranges 6-39320(ten thousand unit) so this unit difference manipulatyes our ml model as they prefer large units but median_income is more important than total_room fro predictions. So to solve this problem we scale all features to similar unit using two methods: min-max scaling and standardization.

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaller = StandardScaler()

housing_num_std_scaled = std_scaller.fit_transform(housing_num)

df_housing_num_std_scaled = pd.DataFrame(housing_num_std_scaled, columns=housing_num.columns, index=housing_num.index)


In [ ]:
housing_num.head()

In [ ]:
df_housing_num_std_scaled.head()

Transformations: transformers are used to tranform an input data to get desired output data. They use parameters calculated by estimators.

For this example we found heavy tails in feature distribution, so to handel then we should make them equally distributed by using log, square root,etc.

So we will build a custom transformer for that.

In [ ]:
#Custom Transformer for log
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log, inverse_func=np.exp, feature_names_out="one-to-one")
log_pop = log_transformer.fit_transform(housing[["population"]])

In [ ]:
log_pop.hist(bins=50, figsize=(4,4))
plt.show()

We have usefull data that is location but it is in terms of longitude and latitude. But we have seen housing prices vary according to locality rather than exact co-ordinates. So giving just co-ordinates to model to predict prices will not be much eifficient so we will use k_means and rbf_kernal to identify each co-ordinates distance from clustered/important location from the data.

A city has some famous location which is highly populated. So houses near those regions will have more value than the far ones.
K_means helps in identifing those famous regions.
rbf_kernal helps in finding distance between each instance and the famous region, gives similarity score (0-1).

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_= KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self
    
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [ ]:
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=42)
similarities = cluster_simil.fit_transform(housing[["latitude", "longitude"]])

longitude = housing[["longitude"]]
latitude = housing[["latitude"]]

plt.scatter(longitude, latitude, c=similarities.max(axis=1), alpha=0.5)
plt.show()

# Yellow highlighted regions are the clusters.

Now lets automate all tranformations using pipelines. Transformation pipelines are build to automate and maintaine the sequence of transformation, esimators used to transform data and provide it to next estimator.

In [ ]:
# pipelines of numeric transformations and categorization of ocean_proximities.
from sklearn.pipeline import make_pipeline, Pipeline

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

cat_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))

In [ ]:
# Now for ratios

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ['ratio']

ratio_pipeline = make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(column_ratio, feature_names_out=ratio_name), StandardScaler())

In [ ]:
#For log 
log_pipeline = make_pipeline(SimpleImputer(strategy="median"), log_transformer)

#Kmeans clusers and rbf similarity.
cluster_simil = ClusterSimilarity(n_clusters=45, gamma=1.0, random_state=42)

In [ ]:
#Now we will automate all these under single tranformer:
from sklearn.compose import ColumnTransformer, make_column_selector

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline, ['total_bedrooms', 'total_rooms']),
    ("rooms_per_house", ratio_pipeline, ['total_rooms', 'households']),
    ("people_per_house", ratio_pipeline, ['population', 'households']),
    ("log", log_pipeline, ['total_bedrooms', 'total_rooms', 'population', 'households', 'median_income']),
    ("geo", cluster_simil, ['latitude', 'longitude']),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object))
], remainder=num_pipeline)

housing_prepared = preprocessing.fit_transform(housing)
housing_prepared.shape

In [ ]:
df_housing_prepared = pd.DataFrame(housing_prepared, columns=preprocessing.get_feature_names_out(), index=housing.index)
df_housing_prepared.head()

We have prepared our final training set that can be provided to ml model to perform efficiently than raw data.

Now lets select most suitable model to train.

In [ ]:
# We will use simple linear model (linear regression)

from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

In [ ]:
#Lets check predictions for training set and compare with labels.

housing_predictions = lin_reg.predict(housing)
housing_predictions[:5].round(-2)

In [ ]:
housing_labels.iloc[:5].values

We can see our first two predictions are way too off but next ones are some what closer to labels. Lets find RMSE value.

In [ ]:
from sklearn.metrics import root_mean_squared_error

lin_rmse = root_mean_squared_error(housing_labels, housing_predictions)
lin_rmse

This is not a good score each district values are differed by an avg of $68975 which is a huge difference. As we can see our model is performing poor itself in training set so this is an example of underfitting. Wethers it got less info to predict(less parameters, features) or we choose very simple model to predict. Before increasing features we will try more powerfull model.

In [ ]:
# Model training
from sklearn.tree import DecisionTreeRegressor

dis_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
dis_reg.fit(housing, housing_labels)

In [ ]:
# Train_set predictions
housing_predictions_dis = dis_reg.predict(housing)
housing_predictions_dis[:5].round(-2)

In [ ]:
# Train_set rsme
dis_rmse = root_mean_squared_error(housing_labels, housing_predictions_dis)
dis_rmse

We got the model with perfect predictions on training set. So features/ limited info wasn't the problem, it was simple model usage as decisiontreeregressor is more pwerfull than linearegressor so it learn't all features and made good predictions. But this doesn't mean its over now just deploy and use model, we have to check wether the model is not overfitted and that can be done using test_set lets analyse performance of the model on test_set. 

In [ ]:
test_housing = strat_test_set.drop("median_house_value", axis=1)
test_housing_labels = strat_test_set["median_house_value"].copy()

In [ ]:
# Test_set predictions
test_housing_predictions_dis = dis_reg.predict(test_housing)
test_housing_predictions_dis[:5].round(-2)

In [ ]:
test_housing_labels.iloc[:5].values

In [ ]:
# Test_set rsme
test_rsme = root_mean_squared_error(test_housing_labels, test_housing_predictions_dis)
test_rsme

We can see it our model performed poor on test_set and got rsme around $106022 which is a lot huge difference. So this is an example of overfitting.
What we can do to solve is lower some features, use less complex model or we can tune hyperparameters by using hold_out_validation technique.

We can do cross_validation without evaluating test_set.

It evaluated using K-splits of train_set
Trained on k-1 splits and evaluated with the remaining split
Such operation is run K times to get the scores.

In [ ]:
from sklearn.model_selection import cross_val_score

dis_rsme = -cross_val_score(dis_reg, housing, housing_labels, scoring= "neg_root_mean_squared_error", cv=10)#applied (-) to convert score to positive

pd.Series(dis_rsme).describe()

Now we can choose a best model but all have same rsme and also they are performing same as linearregressor model so no benefit in using decisiontreeregressor model. We can use RandomForestRegressor and see how it performs.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))
forest_reg.fit(housing, housing_labels)

In [ ]:
forest_housing_predictions = forest_reg.predict(housing)
forest_housing_predictions[:5].round(-2)

In [ ]:
forest_rsme = root_mean_squared_error(housing_labels, forest_housing_predictions)
forest_rsme

We got a decent and lower rsme value compared to linear regression now lets check cross-validation

In [ ]:
forest_rsme = -cross_val_score(forest_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10) #Trains 10 times.
pd.Series(forest_rsme).describe()

We got much better stats than decisiontreeregressor model. We can see rsme on training set is much lower so it is quite still overfitting going on.

So without just wasting time in tweaking the hyperparameters, we got few models lets fine tune them.

For fine tuning we using Grid Search. It takes hyperparameters and test them in cross validation and gives best out of the options.

In [ ]:
from sklearn.model_selection import GridSearchCV

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42))
])

param_grid = [
    {
        'preprocessing__geo__n_clusters':[5,8,10],
        'random_forest__max_features':[4,6,8]
    },
    {
        'preprocessing__geo__n_clusters':[10,15],
        'random_forest__max_features':[6,8,10]
    }
]

grid_search = GridSearchCV(full_pipeline, param_grid=param_grid, cv=3, scoring="neg_root_mean_squared_error")

grid_search.fit(housing, housing_labels) # Trains 45 times.

In [ ]:
# Finidng best hyperparameters.
grid_search.best_params_

In [ ]:
# We got the new estimator which uses 29 features instead of just 24 which were our provided. % new features got added and they contribute in 
#predictions. This new 5 features are due to our n_clusters chenged to 15 as it is best prama.

grid_search.best_estimator_

In [ ]:
cv_rsme = pd.DataFrame(grid_search.cv_results_)
cv_rsme.sort_values(by="mean_test_score", ascending=False, inplace=True)
cv_rsme.head()

We used gridsearch to find best_param but it evaluated only the list of param we gave and fount best out of them so it is irritating to think and give params so instead we can use randomserach which just needs range of params and evaluates them all and dosn't need to train millions of times like gridsearch.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform

param_distribution = {
        'preprocessing__geo__n_clusters': randint(low=3, high=50),
        'random_forest__max_features':randint(low=2, high=20)
    }

rnd_search = RandomizedSearchCV(
    full_pipeline, param_distributions=param_distribution, n_iter=10, cv=3, scoring="neg_root_mean_squared_error", random_state=42
    )

rnd_search.fit(housing, housing_labels)

In [ ]:
rnd_search.best_params_

In [ ]:
rnd_results = pd.DataFrame(rnd_search.cv_results_)
rnd_results.sort_values(by="mean_test_score", ascending=False, inplace=True)
rnd_results.head()

In [ ]:
final_model = rnd_search.best_estimator_

feature_imp=final_model["random_forest"].feature_importances_

In [ ]:
perfect_model_predictions = final_model.predict(housing)
perfect_model_rsme = root_mean_squared_error(housing_labels, perfect_model_predictions)
perfect_model_rsme

In [ ]:
feature_imp.round(2)

we found out impotance of each feature in predicting housing_value. We will use SelectFromModel to analyse importance and remove least important features to see some improvements.

In [ ]:
from sklearn.feature_selection import SelectFromModel

selectfrommodel = SelectFromModel(final_model["random_forest"], prefit=True)

reduced_housing = selectfrommodel.transform(housing_prepared)

In [ ]:
# For selected features.
feature_names = preprocessing.get_feature_names_out()
selected_features = feature_names[selectfrommodel.get_support()]
print(len(selected_features))
print(selected_features)

These above are the important and high importance scored features which will contribute a lot in predictions than others. Now we have reduced_housing we will use it as our final training set to train a new model and check for results(rsme score)

In [ ]:
reduced_housing.shape

reduced_housing_df = pd.DataFrame(reduced_housing, columns=selected_features, index=housing.index)

reduced_housing_df.head()

In [ ]:
#Now train an new model with this reduced/only important featured training set.
best_rf = final_model["random_forest"]

new_model = Pipeline([
    ("preprocessing", preprocessing),
    ("feature_reducer", selectfrommodel),
    ("random_forest", RandomForestRegressor(
    n_estimators=best_rf.n_estimators,
    max_features=best_rf.max_features,
    max_depth=best_rf.max_depth,
    min_samples_leaf=best_rf.min_samples_leaf,
    min_samples_split=best_rf.min_samples_split,
    random_state=42
))
])
new_model.fit(housing, housing_labels)

In [ ]:
new_model_predictions = new_model.predict(housing)
new_model_predictions[:5].round(-2)

In [ ]:
new_model_rsme = root_mean_squared_error(housing_labels, new_model_predictions)
new_model_rsme

In [ ]:
new_model_rsme = -cross_val_score(new_model, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10) #Trains 10 times.
pd.Series(new_model_rsme).describe()

In [ ]:
pd.Series(new_model_rsme).head()

In [ ]:
final_model_rsme = -cross_val_score(final_model, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10) #Trains 10 times.
pd.Series(final_model_rsme).describe()

In [ ]:
test_predictions = final_model.predict(test_housing)

test_predictions_rsme = root_mean_squared_error(test_housing_labels, test_predictions)
test_predictions_rsme

In [ ]:
test_new_model_predictions = new_model.predict(test_housing)
test_new_model_predictions_rsme = root_mean_squared_error(test_housing_labels, test_new_model_predictions)
test_new_model_predictions_rsme

Now we will build and use a Kneighbors custom transformer for countering outliers, smoothing median_income around the nearest instances.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

class IncomeSmoother(BaseEstimator, TransformerMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
    
    def fit(self, X=["longitude", "latitude"], Y="median_income"):
        self.knn = KNeighborsRegressor(n_neighbors=self.n_neighbors)
        self.knn.fit(X, Y)
        return self
    
    def transform(self, X):
        smooths=self.knn.predict(X)
        return np.array(smooths).reshape(-1, 1)

In [ ]:
Median_Income_Smother = IncomeSmoother(n_neighbors=5)
Median_Income_Smother.fit(housing[["longitude", "latitude"]], housing["median_income"])

In [ ]:
polished_median_income = Median_Income_Smother.transform(housing[["longitude", "latitude"]])

In [ ]:
preprocessing2 = ColumnTransformer([
    ("bedrooms", ratio_pipeline, ['total_bedrooms', 'total_rooms']),
    ("rooms_per_house", ratio_pipeline, ['total_rooms', 'households']),
    ("people_per_house", ratio_pipeline, ['population', 'households']),
    ("polished_median_income",Median_Income_Smother, ["median_income"]),
    ("log", log_pipeline, ['total_bedrooms', 'total_rooms', 'population', 'households', 'median_income']),
    ("geo", cluster_simil, ['latitude', 'longitude']),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object))
], remainder=num_pipeline)

In [ ]:
forest_reg = make_pipeline(preprocessing2, RandomForestRegressor(random_state=42))
forest_reg.fit(housing, housing_labels)

In [ ]:
forest_reg_predictions = forest_reg.predict(housing)
forest_reg_rsme = root_mean_squared_error(housing_labels, forest_reg_predictions)
forest_reg_rsme

In [ ]:
test_forest_reg_predictions = forest_reg.predict(test_housing)
test_forest_reg_rsme = root_mean_squared_error(test_housing_labels, test_forest_reg_predictions)
test_forest_reg_rsme

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform

In [ ]:
from sklearn.svm import SVR
from scipy.stats import loguniform


svr_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("svr", SVR())
])

svr_param_distribution = {
    "svr__kernel": ["linear", "rbf"],
    "svr__C": loguniform(1, 100),
    "svr__gamma": loguniform(1, 20)
}

svr_reg = RandomizedSearchCV(
    svr_pipeline, svr_param_distribution, n_iter=10, cv=3, scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1
)

svr_reg.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

In [ ]:
svr_reg.best_params_

In [ ]:
svr_model = svr_reg.best_estimator_

In [ ]:
svr_predictions = svr_model.predict(housing.iloc[:5000])

In [ ]:
from sklearn.metrics import root_mean_squared_error

In [ ]:
svr_rsme = root_mean_squared_error(housing_labels.iloc[:5000], svr_predictions)
svr_rsme

In [ ]:
from sklearn.model_selection import cross_val_score

svr_cross_rmse = -cross_val_score(svr_model, housing.iloc[:5000], housing_labels.iloc[:5000], scoring="neg_root_mean_squared_error", cv=3)
pd.Series(svr_cross_rmse).describe()

In [ ]:
svr_reg.fit(housing.iloc[:10000], housing_labels.iloc[:10000])

In [ ]:
svr_reg.best_params_

In [ ]:
big_svr_model = svr_reg.best_estimator_

In [ ]:
big_svr_predictions = big_svr_model.predict(housing.iloc[:10000])

In [ ]:
big_svr_model_rmse = root_mean_squared_error(housing_labels.iloc[:10000], big_svr_predictions)
big_svr_model_rmse

In [ ]:
big_svr_cross_rmse = -cross_val_score(big_svr_model, housing.iloc[:10000], housing_labels.iloc[:10000], scoring="neg_root_mean_squared_error", cv=3)
pd.Series(big_svr_cross_rmse).describe()